In [1]:
import pandas as pd

from operator import itemgetter
from math import trunc as truncate



In [2]:
# bring in centerlines -> DataFrame centerlines
centerlines_path = "./data/cleaner/centerlines_clean.json"

def read_in_centerlines(path_to_data):
    df = pd.read_json(path_to_data, orient='records', lines=True)
    return df.set_index("OBJECTID")

centerline_data = read_in_centerlines(centerlines_path)
#centerlines.head()

# fix geometry column


In [ ]:
centerline_data

POINT_TYPE = tuple

# might refactor later
def get_geo_ends(df):
    ends = df.GEOMETRY.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})
    df['GEO_start'] = ends.GEOLOW.apply(POINT_TYPE)
    df['GEO_end'] = ends.GEOHI.apply(POINT_TYPE) 
    return df

centerlines = get_geo_ends(centerline_data)

# # set aside full geo for now:
# centerline_GEOMETRY = centerlines.GEOMETRY
# centerlines = centerlines.drop(["GEOMETRY"], axis=1)

#display(centerline_GEOMETRY.head())
centerlines.head()



,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOMETRY,GEO_start,GEO_end
OBJECTID,,,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[[-85.6809503218278, 38.158867088749105], [-85...","(-85.6809503218278, 38.158867088749105)","(-85.68123463490603, 38.158180484404916)"
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[[-85.80120122370631, 38.23056379303306], [-85...","(-85.80120122370631, 38.23056379303306)","(-85.8013692697861, 38.22934335950893)"
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[[-85.80501498815028, 38.228933021550915], [-8...","(-85.80501498815028, 38.228933021550915)","(-85.80483162492965, 38.22753788732054)"
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[[-85.68020545343364, 38.24806713661984], [-85...","(-85.68020545343364, 38.24806713661984)","(-85.67986300781635, 38.24760815991686)"
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[[-85.74203979766622, 38.211814770925905], [-8...","(-85.74203979766622, 38.211814770925905)","(-85.74164752771065, 38.21196859682181)"


In [109]:
GEO = centerlines.GEOMETRY
GEO.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})


,GEOLOW,GEOHI
OBJECTID,,
1,"[-85.6809503218278, 38.158867088749105]","[-85.68123463490603, 38.158180484404916]"
2,"[-85.80120122370631, 38.23056379303306]","[-85.8013692697861, 38.22934335950893]"
3,"[-85.80501498815028, 38.228933021550915]","[-85.80483162492965, 38.22753788732054]"
4,"[-85.68020545343364, 38.24806713661984]","[-85.67986300781635, 38.24760815991686]"
5,"[-85.74203979766622, 38.211814770925905]","[-85.74164752771065, 38.21196859682181]"
...,...,...
180232,"[-85.7085947306709, 38.143298896060436]","[-85.70207255091638, 38.13737907447402]"
180233,"[-85.45093405413414, 38.27043985542058]","[-85.45174798320075, 38.270920967505994]"
180234,"[-85.45174798320075, 38.270920967505994]","[-85.45271378879438, 38.27128541931752]"


In [ ]:

def get_points(geos):
    for geo in geos:
        for point in geo:
            yield point 

def collect_ll(geos):
    longitudes = set()
    latitudes = set()

    for geo in geos:
        for point in geo:
            longitude, latitude = point
            longitudes.add(longitude)
            latitudes.add(latitude)
    
    return longitudes, latitudes

lo, la = collect_ll(GEO)

min_long, max_long = min(lo), max(lo)
min_lat, max_lat = min(la), max(la)

span_long = max_long - min_long
span_lat = max_lat - min_lat 

min_long, span_long, max_long, min_lat, span_lat, max_lat, 

def test_bound(point):
    long, lat = point 
    if not (min_long <= long <= max_long):
        return long, lat
    if not (min_lat <= lat <= max_lat):
        return long, lat

centerlines.GEO_start.apply(test_bound).any() # false test OK 
centerlines.GEO_end.apply(test_bound).any() # false test OK 

PRECISION = 1 << 16

def scale(point):
    long, lat = point 
    long -= min_long
    lat -= min_lat 

    long /= span_long
    lat /= span_lat


    long *= PRECISION
    lat *= PRECISION

    long = truncate(long)
    lat = truncate(lat)

    # if point is on max boundary edge it will equal PRECISION value
    # These points shuold be included in the highest value grid cells for relevant axis
    if long >= PRECISION:
        long = PRECISION - 1

    if lat >= PRECISION:
        lat = PRECISION - 1

    return long, lat

    

spe = centerlines.GEO_end.apply(scale)
sps = centerlines.GEO_start.apply(scale)

def t(p):
    long,lat = p
    if long >= PRECISION:
        return p
    if lat >= PRECISION:
        return p 
    
spe.apply(t).dropna(),sps.apply(t).dropna()

In [ ]:
geo_min = GEO.apply(min)
geo_max = GEO.apply(max)

def agg_long_lat(points, function):
    longi = set()
    lati = set()
    for longitude, latitude in points:
        longi.add(longitude)
        lati.add(latitude)
    return function(longi), function(lati)

min_long, min_lat = agg_long_lat(geo_min, min)
max_long, max_lat = agg_long_lat(geo_max, max)

span_long = abs(max_long - min_long)
span_lat = abs(max_lat - min_lat)

min_long, max_long, span_long, min_lat, max_lat, span_lat

In [53]:


def test_boundary(point):
    long, lat = point 
    return (
        (min_long <= long <= max_long) and 
        (min_lat <= lat <= max_lat))

def tconv(point):
    long, lat = point 
    long -= min_long
    lat -= min_lat
    return long, lat

ends = centerlines.GEO_end
fe = ends[~ends.apply(test_boundary)]

starts = centerlines.GEO_start
fs = starts[~starts.apply(test_boundary)]

fe, fs

fe.apply(tconv),fs.apply(tconv)

plat = fe[1334][1]

plat

38.00058434740185

In [ ]:
def scale(point):
    longitude, latitude = point 
    longitude = (longitude - min_long) / span_long
    latitude = (latitude - min_lat) / span_lat
    return longitude, latitude

PRECISION = 1 << 16

def cast(pt):
    long, lat = pt 
    long = truncate(long * PRECISION).to_bytes(length=2)
    lat = truncate(lat * PRECISION).to_bytes(length=2)
    return long, lat

def test(pt):
    a, b = pt
    if (a < 0) or (b < 0):
        return pt

centerlines.GEO_.apply(scale).apply(test).dropna()


Series([], Name: GEO_start, dtype: object)

In [ ]:
# def iter_points():
#     for centerline in centerlines.GEOMETRY:
#         for point in centerline:
#             yield point 

# infinity = float('inf')

# def find_box():
#     max_longitude = -infinity
#     min_longitude = infinity
#     max_latitude = -infinity
#     min_latitude = infinity 

#     for centerline in centerlines.GEOMETRY:
#         for longitude, latitude in centerline:
#             if longitude > max_longitude:
#                 max_longitude = longitude
#             if longitude < min_longitude:
#                 min_longitude = longitude
#             if latitude > max_latitude:
#                 max_latitude = latitude
#             if latitude < min_latitude:
#                 min_latitude = latitude 

#     return {'max_longitude': max_longitude, 'min_longitude':min_longitude,
#             'max_latitude': max_latitude, 'min_latitude':min_latitude}


# box = find_box()

# min_longitude = box['min_longitude']
# longitude_span = box['max_longitude'] - min_longitude

# min_latitude = box['min_latitude']
# latitude_span = box['max_latitude'] - min_latitude


0.4204147651767453


OverflowError: int too big to convert

In [ ]:

PRECISION = 2 ** 16

def convert_point(point):
    longitude, latitude = point 
    longitude = (longitude - min_longitude) / longitude_span
    try:
        longitude = truncate(longitude * PRECISION)#.to_bytes()#(length=2)
    except:
        print(longitude)
        raise
    try:
        latitude = (latitude - min_latitude) / latitude_span
        latitude = truncate(latitude * PRECISION).to_bytes()
    except:
        print(latitude)
        raise# (length=2)
    return longitude , latitude



centerlines.GEO_start.apply(convert_point)

box

In [44]:
min_longitude = -85.94712712079293
max_longitude = -85.3443621648922

min_latitude = 37.99712528351634 
max_latitude = 38.38023822809115

longitude_span = max_longitude - min_longitude
latitude_span = max_latitude - min_latitude 

PRECISION = 2 ** 16

def encode_point(point):
    longitude, latitude = point 
    longitude = (longitude - min_longitude) / longitude_span
    longitude = truncate(longitude * PRECISION).to_bytes(length=2)
    latitude = (latitude - min_latitude) / latitude_span 
    latitude = truncate(latitude * PRECISION).to_bytes(length=2)
    return longitude + latitude 


In [16]:
centerlines.GEO_start.apply(convert_point)



OBJECTID
1         (28651, 27552)
2         (15473, 40032)
3         (15055, 39748)
4         (28732, 43079)
5         (21956, 36768)
               ...      
180232    (25621, 24842)
180233    (53857, 46973)
180234    (53768, 47057)
180235    (53662, 47120)
180236    (53768, 47057)
Name: GEO_start, Length: 35039, dtype: object

In [17]:
centerlines.GEO_end.apply(convert_point)


OBJECTID
1         (28620, 27432)
2         (15455, 39820)
3         (15075, 39505)
4         (28770, 42999)
5         (21999, 36795)
               ...      
180232    (26336, 23811)
180233    (53768, 47057)
180234    (53662, 47120)
180235    (53857, 46973)
180236    (53895, 47210)
Name: GEO_end, Length: 35039, dtype: object

In [ ]:
# centerlines.GEO_start.apply(encode_point).apply(bytes.hex) # no problem

def scale(point):
    long, lat = point 
    long -= min_longitude
    long /= latitude_span

    lat -= min_latitude
    lat /= latitude_span

    if (long > 1) or (lat > 1):
        return (long, lat)
        
long_too_big = centerlines.GEO_end.apply(scale).dropna()


In [60]:

l2b = centerlines.loc[long_too_big.index]
l2b
sf = ((l2b.GEO_end.apply(itemgetter(0)) - box['min_longitude']) / longitude_span) * PRECISION
sf.apply(truncate)

#2**16 # 65535

OBJECTID
7         41281
13        54017
15        57907
21        45869
22        54325
          ...  
180230    43419
180233    53768
180234    53662
180235    53857
180236    53895
Name: GEO_end, Length: 5909, dtype: int64